In [1]:
import pandas as pd

flights = pd.read_csv('../data/cleaned/flights_cleaned.csv')
active_flights = pd.read_csv('../data/processed/active_flights.csv')
cancelled_flights = pd.read_csv('../data/processed/cancelled_flights.csv')

In [2]:
kpi_summary = pd.DataFrame({
    'Metric': [
        'Total Flights',
        'Cancelled Flights',
        'Cancellation Rate (%)',
        'Average Arrival Delay',
        'Average Departure Delay',
        'ML Accuracy (%)'
    ],
    'Value': [
        len(flights),
        len(cancelled_flights),
        round(flights['CANCELLED'].mean() * 100, 2),
        round(active_flights['ARR_DELAY'].mean(), 2),
        round(active_flights['DEP_DELAY'].mean(), 2),
        99
    ]
})

kpi_summary

,Metric,Value
0,Total Flights,3000000.00
1,Cancelled Flights,79140.00
2,Cancellation Rate (%),2.64
3,Average Arrival Delay,4.26
4,Average Departure Delay,10.10
5,ML Accuracy (%),99.00


In [3]:
kpi_summary.to_csv(
    '../dashboard_data/kpi_summary.csv',
    index=False
)

In [4]:
monthly_delay = (
    active_flights.groupby('MONTH')['ARR_DELAY']
    .mean()
    .reset_index()
)

monthly_delay.columns = [
    'Month',
    'Average_Arrival_Delay'
]

monthly_delay

,Month,Average_Arrival_Delay
0,1,2.190728
1,2,3.712646
2,3,2.255956
3,4,3.539976
4,5,3.359411
5,6,10.061753
6,7,9.491415
7,8,6.453106
8,9,-0.512181
9,10,1.424036


In [5]:
monthly_delay.to_csv(
    '../dashboard_data/monthly_delay.csv',
    index=False
)

In [6]:
delay_causes = pd.DataFrame({
    'Delay_Type': [
        'Carrier Delay',
        'Weather Delay',
        'NAS Delay',
        'Security Delay',
        'Late Aircraft Delay'
    ],
    'Total_Delay_Minutes': [
        active_flights['DELAY_DUE_CARRIER'].sum(),
        active_flights['DELAY_DUE_WEATHER'].sum(),
        active_flights['DELAY_DUE_NAS'].sum(),
        active_flights['DELAY_DUE_SECURITY'].sum(),
        active_flights['DELAY_DUE_LATE_AIRCRAFT'].sum()
    ]
})

delay_causes

delay_causes.to_csv(
    '../dashboard_data/delay_causes.csv',
    index=False
)

In [7]:
def classify_delay(delay):
    if delay <= 15:
        return 'On Time'
    elif delay <= 60:
        return 'Moderate Delay'
    else:
        return 'Severe Delay'


active_flights['DELAY_CATEGORY'] = (
    active_flights['ARR_DELAY']
    .apply(classify_delay)
)

delay_severity = (
    active_flights['DELAY_CATEGORY']
    .value_counts()
    .reset_index()
)

delay_severity.columns = [
    'Delay_Category',
    'Flight_Count'
]

delay_severity

delay_severity.to_csv(
    '../dashboard_data/delay_severity.csv',
    index=False
)

In [8]:
top_delayed_airlines = (
    active_flights.groupby('AIRLINE')['ARR_DELAY']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_delayed_airlines.columns = [
    'Airline',
    'Average_Arrival_Delay'
]

top_delayed_airlines

top_delayed_airlines.to_csv(
    '../dashboard_data/top_delayed_airlines.csv',
    index=False
)

In [9]:
busiest_airports = (
    flights['ORIGIN']
    .value_counts()
    .head(10)
    .reset_index()
)

busiest_airports.columns = [
    'Airport',
    'Total_Flights'
]

busiest_airports

busiest_airports.to_csv(
    '../dashboard_data/busiest_airports.csv',
    index=False
)

In [10]:
active_flights['ROUTE'] = (
    active_flights['ORIGIN']
    + ' → ' +
    active_flights['DEST']
)

route_stats = (
    active_flights.groupby('ROUTE')
    .agg(
        Average_Arrival_Delay=('ARR_DELAY', 'mean'),
        Total_Flights=('ROUTE', 'count')
    )
    .reset_index()
)

route_stats = route_stats[
    route_stats['Total_Flights'] > 100
]

top_delayed_routes = (
    route_stats
    .sort_values('Average_Arrival_Delay', ascending=False)
    .head(10)
)

top_delayed_routes

top_delayed_routes.to_csv(
    '../dashboard_data/top_delayed_routes.csv',
    index=False
)

In [11]:
taxi_out_congestion = (
    active_flights.groupby('ORIGIN')['TAXI_OUT']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

taxi_out_congestion.columns = [
    'Airport',
    'Average_Taxi_Out_Time'
]

taxi_out_congestion

taxi_out_congestion.to_csv(
    '../dashboard_data/taxi_out_congestion.csv',
    index=False
)

In [12]:
ml_metrics = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score'
    ],
    'Value': [
        0.99,
        0.97,
        0.98,
        0.98
    ]
})

ml_metrics

ml_metrics.to_csv(
    '../dashboard_data/ml_metrics.csv',
    index=False
)